# just_tx vs just_tx_hal：处理 PPS

同一块 ESP32-C3、同一 AP、同一台 TR3000。比较两条**发送**路径在满速刺激下能处理多少包。
计的是设备侧进入回发路径的包，**不是** RTT，也**不是**主机提交速率。RTT / ~5% 丢包见 `just_tx_rtt.ipynb`。

| 臂 | 固件 / 冷表 | 路径 | 计数器 |
|---|---|---|---|
| **HAL** | `just_tx_hal` | 应用层 UDP:8080 原载荷回发 | 串口 `rxcnt 17`（`send_to` 成功才加） |
| **XDP** | `just_drop` 固件 + 冷表 `just_tx.elf` | Wi-Fi 入口 MAC 交换后 `XDP_TX` | map `rxcnt[17]`（进 TX 路径就加） |

**刺激**：TR3000 `udp_blast` 满速小 UDP（无 seq）。`sender_target_pps` 只记录主机提交速率。

主图只画 **Mean processing PPS**（轮间 Student *t* 误差线）。1 s 相对 ADEV 只作表，不进图。

数据：`formal.experiment_*`，`run_kind=pilot`，`source_note = xpost-test just-tx-pps`。

**复现**（cwd = `xpost/`）：

```bash
cargo xtask agent --listen 0.0.0.0:9240 --no-monitor
```

TR3000：

```bash
XPOST_AGENT_ENDPOINT=192.168.110.96:9240 ./xpost-test just-tx-pps
```

本笔记只读库。


In [ ]:
from collections import defaultdict

import matplotlib.pyplot as plt
from IPython.display import display

from utils.db import dsn_summary
from utils.experiment import (
    figure_tx_pps_mean,
    load_counter_samples,
    load_runs,
    mean_across_runs,
    mean_pps,
    mean_t_ci,
    overlapping_relative_adev,
    summarize_variant,
)

SOURCE_NOTE = "xpost-test just-tx-pps"

plt.ioff()
plt.close("all")
print(dsn_summary())

runs = load_runs(run_kind="pilot", source_note=SOURCE_NOTE)
assert len(runs), (
    "formal.experiment_runs 无 just-tx-pps pilot 行：先跑 ./xpost-test just-tx-pps"
)
samples = load_counter_samples(runs["run_id"].tolist())
print(f"runs={len(runs)} samples={len(samples)}")

by_variant = defaultdict(list)
adev_runs = defaultdict(list)

for _, row in runs.iterrows():
    sub = samples[samples["run_id"] == row["run_id"]].sort_values("time_ns")
    t_ns = sub["time_ns"].tolist()
    c = sub["counter_value"].tolist()
    variant = row["variant"]
    by_variant[variant].append(
        mean_pps(t_ns, c, run_id=row["run_id"], variant=variant, pair_id=int(row["pair_id"]))
    )
    adev = overlapping_relative_adev(t_ns, c)
    if adev:
        adev_runs[variant].append({tau: d["adev"] for tau, d in adev.items()})

summaries = {v: summarize_variant(rs) for v, rs in by_variant.items()}
adev_pool = {v: mean_across_runs(rows) for v, rows in adev_runs.items()}


def _at(pool, tau=1.0):
    row = pool.get(tau) or {}
    return row.get("mean"), (row.get("lo"), row.get("hi"))


for v, s in summaries.items():
    a_mean, (a_lo, a_hi) = _at(adev_pool.get(v, {}))
    print(
        f"{v}: valid={s['n_valid']}/{s['n_total']}  "
        f"PPS={s['mean_pps']:.0f}  ADEV@1s={100 * a_mean:.2f}% "
        f"({100 * a_lo:.2f}–{100 * a_hi:.2f})  "
        f"per-run={[round(x) for x in s['per_run_pps']]}"
    )

hal, xdp = summaries.get("hal", {}), summaries.get("xdp", {})
hal_pps = hal.get("per_run_pps") or []
xdp_pps = xdp.get("per_run_pps") or []
_, hal_pps_lo, hal_pps_hi = mean_t_ci(hal_pps)
_, xdp_pps_lo, xdp_pps_hi = mean_t_ci(xdp_pps)

fig, _ = figure_tx_pps_mean(
    hal_mean_pps=hal.get("mean_pps", float("nan")),
    xdp_mean_pps=xdp.get("mean_pps", float("nan")),
    hal_pps_ci=(hal_pps_lo, hal_pps_hi),
    xdp_pps_ci=(xdp_pps_lo, xdp_pps_hi),
    title="just_tx HAL vs XDP",
)
display(fig)
plt.close(fig)

print("\nADEV table (paper; not plotted):")
print(f"{'variant':8} {'PPS':>8} {'ADEV@1s':>10} {'95% CI':>22} {'n':>3}")
for v, s in summaries.items():
    a_mean, (a_lo, a_hi) = _at(adev_pool.get(v, {}))
    print(
        f"{v:8} {s['mean_pps']:8.0f} {100 * a_mean:9.2f}% "
        f"({100 * a_lo:6.2f}–{100 * a_hi:6.2f})% {s['n_valid']:3d}"
    )
